# Construction d'un Cube de Synthèse d'Assurance des Revenus Télécom avec PROC SUMMARY

## Synthèse

Une équipe d'assurance des revenus d'un opérateur mobile pré-agrège un mois d'enregistrements de facturation par abonné-jour dans un cube de synthèse compact afin que les analystes puissent explorer le chiffre d'affaires réglé par région, palier tarifaire et type d'appel sans réanalyser la table de faits brute. `PROC SUMMARY` regroupe 100 enregistrements abonné-jour en un ensemble multidimensionnel de cellules : la granularité la plus fine (région x palier x type d'appel) remplit 25 des 27 cellules possibles, tandis que des sous-cubes nommés fournissent les agrégats que les analystes interrogent le plus. Pour ce mois échantillon, l'opérateur a réglé 222,78 $ à travers les trois régions actives, Sud (97,44 $) et Est (86,94 $) représentant ensemble 83% du chiffre d'affaires et Nord à la traîne avec 38,40 $. Le sous-cube le plus riche est le service Voix du palier Plus (59,06 $ sur 18 enregistrements), et le classement des cellules par rendement par enregistrement fait apparaître les cellules de données comme les cibles les plus intéressantes pour un audit de fuite de revenus (meilleur rendement 7,87 $/enregistrement). Chaque chiffre ci-dessous est lu directement à partir de la sortie exécutée.

## Sources de Données

| Jeu de données | Lignes | Granularité | Variables clés |
|---------|------|-------|---------------|
| `work.cdr_billing` | 100 | Une ligne par résumé d'usage abonné-jour | `region` (Est/Sud/Nord), `plan_tier` (Prépayé/Basique/Plus), `call_type` (Voix/SMS/Données), `device_class`, `bill_month`, `revenue`, `call_minutes`, `data_mb`, `subscriber_wt` |
| `work.cube_nway` | 25 | Une ligne par cellule non vide (région x plan_tier x call_type) | `_FREQ_`, `rev_total`, `rev_mean`, `rev_max`, `min_total`, `data_total` |
| `work.cube_hier` | 22 | Une ligne par cellule des sous-cubes nommés d'exploration | `_TYPE_`, `_FREQ_`, `rev_total` |

Toutes les données sont générées en ligne avec `call streaminit()` / `rand()` ; aucun fichier externe ni accès réseau n'est requis. Cet environnement fonctionne sans licence, donc les jeux de données écrits sont plafonnés à 100 observations — le scénario est dimensionné de sorte que le cube soit entièrement peuplé dans cette limite.

## Des enregistrements bruts d'appels détaillés à un cube explorable

Les opérateurs mobiles règlent le chiffre d'affaires sur des millions d'événements d'usage quotidiens. Pour permettre aux analystes de l'assurance des revenus de répondre à des questions comme *« Quel a été le chiffre d'affaires voix du palier Plus dans la région Sud le mois dernier ? »* sans réanalyser la table de faits brute à chaque fois, nous **pré-agrégons** les données dans un cube de synthèse compact.

`PROC SUMMARY` est l'outil de référence de Base SAS pour exactement ce besoin : il regroupe une table de faits plate selon une ou plusieurs dimensions `CLASS` et écrit les statistiques demandées dans un jeu de données de sortie, en marquant chaque ligne avec `_TYPE_` (quelles dimensions sont actives) et `_FREQ_` (nombre d'enregistrements derrière la cellule). Ce jeu de données de sortie *est* un cube multidimensionnel — la même structure d'agrégation qu'exposerait un outil OLAP, matérialisée sous la forme d'un jeu de données SAS ordinaire que l'on peut imprimer, joindre et découper.

Ce notebook :

1. Génère un mois réaliste d'enregistrements de facturation par abonné-jour.
2. Construit le cube à la granularité la plus fine (région x palier tarifaire x type d'appel) avec `PROC SUMMARY NWAY`.
3. Matérialise des **sous-cubes d'exploration** nommés avec l'instruction `TYPES`.
4. Projette le chiffre d'affaires sur la base d'abonnés avec un `WEIGHT`, explore une région, et classe les cellules par rendement par enregistrement pour le triage des fuites.

## Étape 1 - Générer des données synthétiques d'appels détaillés / de facturation

Chaque ligne résume l'usage d'un service par un abonné pour un jour donné. Nous utilisons `call streaminit` pour la reproductibilité et `rand()` pour tirer des distributions plausibles : le chiffre d'affaires et l'usage varient selon le palier tarifaire, le chiffre d'affaires voix suit les minutes facturables, et le chiffre d'affaires données suit les mégaoctets. Chaque `RAND('table', ...)` reçoit une probabilité par catégorie afin que chaque région, palier et type d'appel apparaissent dans l'échantillon de 100 enregistrements. Un petit poids d'enquête `subscriber_wt` est attaché pour pouvoir démontrer une mesure pondérée plus loin.

In [1]:
données work.cdr_billing;
    appeler streaminit(20260131);
    longueur region $6 plan_tier $12 call_type $9 device_class $12 bill_month $7;
    étiquette revenue       = "Chiffre d'Affaires Réglé (USD)"
          call_minutes  = "Minutes Vocales Facturables"
          data_mb       = "Volume de Données (Mo)"
          subscriber_wt = "Poids d'Enquête Abonné";

    faire i = 1 jusqu_à 100;
        /* --- Dimensions : une probabilité par niveau, somme = 1.0 --- */
        r = rand("table", 0.40, 0.33, 0.27);
        sélectionner (r);
            quand (1) region = "Est";
            quand (2) region = "Sud";
            autrement region = "Nord";
        fin;

        p = rand("table", 0.30, 0.40, 0.30);
        sélectionner (p);
            quand (1) plan_tier = "Prépayé";
            quand (2) plan_tier = "Basique";
            autrement plan_tier = "Plus";
        fin;

        c = rand("table", 0.50, 0.30, 0.20);
        sélectionner (c);
            quand (1) call_type = "Voix";
            quand (2) call_type = "SMS";
            autrement call_type = "Données";
        fin;

        si rand("uniform") < 0.55 alors device_class = "Smartphone";
        sinon device_class = "Classique";

        bill_month = "2026-01";

        /* --- Mesures, mises à l'échelle par palier et service --- */
        tier_mult = (plan_tier = "Prépayé")*0.7
                  + (plan_tier = "Basique")*1.0
                  + (plan_tier = "Plus")*1.7;

        call_minutes = round((call_type = "Voix")
                       * rand("gamma", 2.0) * 18 * tier_mult, 0.1);
        data_mb      = round((call_type = "Données")
                       * rand("gamma", 1.5) * 220 * tier_mult, 0.1);

        base_rev = 0.05*call_minutes + 0.010*data_mb
                 + (call_type = "SMS") * rand("poisson", 30) * 0.03;
        revenue = round(base_rev * (0.85 + 0.30*rand("uniform")), 0.01);

        subscriber_wt = round(0.8 + rand("uniform")*1.6, 0.01);

        sortie;
    fin;
    supprimer i r p c tier_mult base_rev;
exécuter;

procédure print données=work.cdr_billing(obs=8) étiquette noobs;
    étiquette region="Région" plan_tier="Palier" call_type="Type d'Appel"
          device_class="Type d'Appareil" bill_month="Mois de Facturation";
    titre "Échantillon d'Enregistrements de Facturation par Abonné-Jour";
exécuter;

                              Échantillon d'Enregistrements de Facturation par Abonné-Jour                              

 Région     Palier  Type d'Appel  Type d'Appareil  Mois de Facturation  Minutes Vocales Facturables   Volume de Données (Mo)    Chiffre d'Affaires Réglé (USD)    Poids d'Enquête Abonné
Nord     Plus       SMS           Smartphone       2026-01                                        0                        0                              0.67                      1.13
Sud      Prépayé    SMS           Classique        2026-01                                        0                        0                              0.94                      1.42
Sud      Plus       SMS           Smartphone       2026-01                                        0                        0                              0.99                      0.86
Sud      Plus       SMS           Smartphone       2026-01                                        0                        0              


NOTE: DATA work.cdr_billing


NOTE: Wrote work.cdr_billing (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds
NOTE: PROC PRINT data=work.cdr_billing

NOTE: PROC PRINT completed: 8 observations printed, 9 variables


## Étape 2 - Construire le cube à la granularité la plus fine avec PROC SUMMARY NWAY

`NWAY` ne conserve que la combinaison la plus détaillée de toutes les dimensions `CLASS` - ici chaque cellule peuplée (région x plan_tier x call_type). Pour chaque cellule, nous stockons le `SUM`, la `MEAN`, et le `MAX` du chiffre d'affaires plus le total des minutes et des mégaoctets, afin qu'un analyste puisse lire le chiffre d'affaires total par cellule, en déduire la moyenne par enregistrement, et repérer la plus grosse charge unique. `_FREQ_` enregistre combien d'abonné-jours se cachent derrière chaque cellule. Nous supprimons `_TYPE_` ici car, à la granularité NWAY, chaque ligne a le même type.

In [2]:
procédure summary données=work.cdr_billing nway;
    classe region plan_tier call_type;
    var revenue call_minutes data_mb;
    sortie out=work.cube_nway(supprimer=_type_)
           sum(revenue)=rev_total  mean(revenue)=rev_mean  max(revenue)=rev_max
           sum(call_minutes)=min_total
           sum(data_mb)=data_total;
exécuter;

procédure print données=work.cube_nway(obs=12) noobs étiquette;
    étiquette region="Région" plan_tier="Palier" call_type="Type d'Appel"
          rev_total="CA Total" rev_mean="CA Moyen" rev_max="CA Max"
          min_total="Minutes Totales" data_total="Données Totales (Mo)";
    titre "Cellules du Cube NWAY (région * palier * type d'appel)";
    format rev_total rev_mean rev_max min_total data_total comma10.2;
exécuter;

                                 Cellules du Cube NWAY (région * palier * type d'appel)                                 

 Région     Palier  Type d'Appel  _FREQ_  CA Total  CA Moyen  CA Max  Minutes Totales   Données Totales (Mo)
Est      Basique    Données            4     18.17      4.54    8.05             0.00               1,807.90
Est      Basique    SMS                4      4.07      1.02    1.24             0.00                   0.00
Est      Basique    Voix               7     15.08      2.15    3.73           302.50                   0.00
Est      Plus       Données            1      5.54      5.54    5.54             0.00                 573.90
Est      Plus       SMS                4      3.59      0.90    0.95             0.00                   0.00
Est      Plus       Voix               7     23.87      3.41    8.01           491.70                   0.00
Est      Prépayé    SMS                3      3.00      1.00    1.06             0.00                   0.00
Est   


NOTE: PROC MEANS
NOTE: Output dataset work.cube_nway has 25 observations and 9 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 12 observations printed, 9 variables


## Étape 3 - Matérialiser des sous-cubes d'exploration nommés avec TYPES

Un cube OLAP pré-stocke les agrégations que les analystes consultent le plus. L'instruction `TYPES` fait exactement cela : chaque terme demande à `PROC SUMMARY` d'émettre un sous-cube. Nous demandons le total général `()`, la marginale `region`, et les sous-cubes à deux dimensions `region*plan_tier` et `call_type*plan_tier` - les chemins d'exploration qu'exposerait un tableau de bord de revenus. SAS marque chaque sous-cube avec un code `_TYPE_` (un masque de bits sur la liste `CLASS`), si bien qu'un seul jeu de données de sortie porte tous les niveaux de la hiérarchie.

In [3]:
procédure summary données=work.cdr_billing;
    classe region plan_tier call_type;
    var revenue;
    types () region region*plan_tier call_type*plan_tier;
    sortie out=work.cube_hier
           sum(revenue)=rev_total;
exécuter;

procédure print données=work.cube_hier noobs étiquette;
    titre "Agrégations de Sous-Cubes : Total Général, Région, Région*Palier, TypeAppel*Palier";
    var _type_ region plan_tier call_type _freq_ rev_total;
    étiquette region="Région" plan_tier="Palier" call_type="Type d'Appel"
          rev_total="CA Total";
    format rev_total comma10.2;
exécuter;

                   Agrégations de Sous-Cubes : Total Général, Région, Région*Palier, TypeAppel*Palier                   

_TYPE_   Région     Palier  Type d'Appel  _FREQ_  CA Total
     0                                       100    222.78
     3           Basique    Données            8     38.06
     3           Basique    SMS                8      8.03
     3           Basique    Voix              20     42.33
     3           Plus       Données            3     17.46
     3           Plus       SMS               13     11.75
     3           Plus       Voix              18     59.06
     3           Prépayé    Données            7     14.50
     3           Prépayé    SMS                7      6.82
     3           Prépayé    Voix              16     24.77
     4  Est                                   38     86.94
     4  Nord                                  23     38.40
     4  Sud                                   39     97.44
     6  Est      Basique                      15    


NOTE: PROC MEANS
NOTE: Output dataset work.cube_hier has 22 observations and 6 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_hier

NOTE: PROC PRINT completed: 22 observations printed, 6 variables


## Étape 4 - Projection pondérée, exploration régionale, et triage des fuites

Trois lectures qu'une équipe d'assurance des revenus effectue réellement sur le cube :

- **Projection pondérée.** Attacher `WEIGHT=subscriber_wt` à un résumé `region*plan_tier` rapporte un chiffre d'affaires mis à l'échelle de la base d'abonnés complète que l'échantillon représente, plutôt que les lignes brutes échantillonnées.
- **Exploration.** Filtrer le cube NWAY sur une région développe une seule branche de la hiérarchie - ici Sud - vers son détail palier par service.
- **Triage des fuites.** Trier les cellules par chiffre d'affaires moyen par enregistrement fait apparaître les cellules à plus haut rendement ; les cellules à faible fréquence et haut rendement sont exactement ce qu'un audit scrute pour des revenus mal tarifés ou en fuite.

In [4]:
/* Chiffre d'affaires pondéré, projeté sur la base d'abonnés */
procédure summary données=work.cdr_billing nway;
    classe region plan_tier;
    var revenue;
    poids subscriber_wt;
    sortie out=work.cube_wt(supprimer=_type_ _freq_)
           sum(revenue)=rev_weighted  n(revenue)=records;
exécuter;

procédure print données=work.cube_wt noobs étiquette;
    titre "Chiffre d'Affaires Pondéré par Région * Palier (Projeté sur la Base d'Abonnés)";
    étiquette region="Région" plan_tier="Palier" rev_weighted="CA Pondéré" records="Nb. Enregistrements";
    format rev_weighted comma10.2;
exécuter;

/* Explorer la branche region Sud du cube */
procédure print données=work.cube_nway noobs étiquette;
    où region = "Sud";
    titre "Exploration de Sud : Cellules de Chiffre d'Affaires par Palier et Type d'Appel";
    var plan_tier call_type _freq_ rev_total rev_mean;
    étiquette plan_tier="Palier" call_type="Type d'Appel" rev_total="CA Total" rev_mean="CA Moyen";
    format rev_total rev_mean comma10.2;
exécuter;

/* Classer les cellules par rendement par enregistrement pour le triage des fuites */
procédure sort données=work.cube_nway out=work.cube_ranked;
    par descendant rev_mean;
exécuter;

procédure print données=work.cube_ranked(obs=6) noobs étiquette;
    titre "Cellules à Chiffre d'Affaires Moyen le Plus Élevé (Rendement par Enregistrement)";
    var region plan_tier call_type _freq_ rev_mean rev_max;
    étiquette region="Région" plan_tier="Palier" call_type="Type d'Appel"
          rev_mean="CA Moyen" rev_max="CA Max";
    format rev_mean rev_max comma10.2;
exécuter;

                     Chiffre d'Affaires Pondéré par Région * Palier (Projeté sur la Base d'Abonnés)                     

 Région     Palier    CA Pondéré  Nb. Enregistrements
Est      Basique           50.85                   15
Est      Plus              59.59                   12
Est      Prépayé           29.77                   11
Nord     Basique           18.30                    7
Nord     Plus              22.42                    7
Nord     Prépayé           20.56                    9
Sud      Basique           58.63                   14
Sud      Plus              56.29                   15
Sud      Prépayé           27.77                   10

                     Exploration de Sud : Cellules de Chiffre d'Affaires par Palier et Type d'Appel                     

   Palier  Type d'Appel  _FREQ_  CA Total  CA Moyen
Basique    Données            3     12.02      4.01
Basique    SMS                2      2.01      1.00
Basique    Voix               9     22.51      2.50
Plus   


NOTE: PROC MEANS
NOTE: Output dataset work.cube_wt has 9 observations and 4 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_wt

NOTE: PROC PRINT completed: 9 observations printed, 4 variables
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 9 observations printed, 5 variables
NOTE: PROC SORT data=work.cube_nway

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 25 rows from work.cube_nway.
NOTE: Wrote work.cube_ranked (25 rows, 9 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=work.cube_ranked

NOTE: PROC PRINT completed: 6 observations printed, 6 variables


## Interprétation des résultats

Le cube de synthèse transforme 100 lignes brutes d'abonné-jour en un ensemble compact de cellules pré-agrégées qui répondent instantanément aux questions d'exploration, sans réanalyser la table de faits :

- **Où se trouve le chiffre d'affaires.** La marginale `region` (`_TYPE_=4`) montre que Sud a réglé 97,44 $ et Est 86,94 $ - ensemble 83% des 222,78 $ du mois - tandis que Nord a contribué 38,40 $. Dans le sous-cube `call_type*plan_tier` (`_TYPE_=3`), le service Voix du palier Plus est la cellule la plus riche à 59,06 $ sur 18 enregistrements, avec le service Voix du palier Basique juste derrière à 42,33 $.
- **Projection pondérée.** Appliquer le poids d'enquête redistribue le classement vers les paliers dont les abonnés portent plus de poids : Est-Plus (59,59 $) et Sud-Basique (58,63 $) sont en tête du chiffre d'affaires projeté `region*plan_tier`, une image différente des totaux régionaux non pondérés et un rappel qu'il faut rapporter le chiffre d'affaires projeté plutôt qu'échantillonné pour dimensionner l'exposition.
- **Rendement par enregistrement et triage des fuites.** Classer les cellules NWAY par `rev_mean` place les cellules à usage de données en tête - Nord-Basique-Données (7,87 $/enregistrement) et Sud-Plus-Données (5,96 $/enregistrement) - confirmant que l'usage intensif de données génère le chiffre d'affaires par enregistrement le plus élevé. Les leaders à faible fréquence (un ou deux enregistrements) sont précisément les cellules pour lesquelles un analyste de l'assurance des revenus irait chercher les enregistrements d'appels détaillés sous-jacents, pour confirmer que la charge élevée est correctement tarifée plutôt qu'une erreur.

Pour une équipe d'assurance des revenus, ce cube est le fondement des tableaux de bord de variance : comparer le chiffre d'affaires réglé au chiffre d'affaires attendu de la grille tarifaire par cellule (région, palier, type d'appel), et les cellules dont la moyenne ou le total pondéré divergent le plus deviennent les cas de fuite à auditer. Comme toute la structure est un jeu de données SAS ordinaire, le cube du mois suivant peut être réuni, différencié, ou joint à une grille tarifaire avec les mêmes outils Base SAS.